# LOKLOK Qualitative RAG Pipeline (Colab, T4)

This notebook implements the **frozen, local, RAG-grounded qualitative coding and validation plan** for the LOKLOK thesis. It is a scaffold for the final thesis: it does **not** modify the thesis and it does **not** run automatically.

**Non-negotiable rules**

1. No outside text is sent to an API; models run locally.
2. Every code must carry a verbatim quote from the target segment. Failed quotes are dropped and counted.
3. A human gold set is frozen **before** coding. Dev and gold are disjoint.
4. PO is excluded unless `03_data/PROVENANCE_PO.md` establishes researcher-written field notes.
5. The integration check is a **prior-reweighted sensitivity** on the existing 256-point sweep, not a new ABM run.
6. A null result is reported as a null result.

**Stages**

- **Q1**: ingest, split, gold template, retrieval index.
- **Q2**: fresh runtime; local Qwen 7B coding with resume; quote verification.
- **Q3**: human-gold validation, prevalence, prior-reweighted sweep, LaTeX macros.

Run Q1 first, label `gold_template.csv`, save it as `gold_labels.csv`, freeze it, then run Q2 and Q3. Use a fresh Colab runtime for Q2.

## Q1. Install the light dependencies

In [ ]:
!pip -q install pandas numpy pyarrow sentence-transformers rank_bm25 rapidfuzz scikit-learn pyyaml

## Q1. Locate the package

In [ ]:
from pathlib import Path
import os, sys, json, zipfile
ROOT = Path("/content/loklok_v8")
if not ROOT.exists():
    try:
        from google.colab import files
        print("Upload colab_bundle_qual.zip")
        up = files.upload()
        for name in up:
            with zipfile.ZipFile(name) as z:
                z.extractall("/content")
    except Exception as exc:
        print("Upload/extract unavailable:", exc)
if not ROOT.exists():
    ROOT = Path("/content/loklok-thesis-final-upgrade")
print("ROOT:", ROOT, ROOT.exists())
assert ROOT.exists()
sys.path.insert(0, str(ROOT / "02_code/qual"))
from qual_pipeline import load_config
CFG = load_config(ROOT / "02_code/qual/qual_config.yaml")
CFG["_root"] = str(ROOT)
print(CFG["primary_model"], CFG["use_po"], CFG["_root"])

## Q1. Ingest, split, gold template, and index

In [ ]:
from qual_pipeline import ingest, make_splits, make_gold_template, build_index, po_gate
GATE = po_gate(ROOT, CFG)
print("PO gate:", GATE)
CORPUS = ingest(ROOT, CFG)
SPLITS = make_splits(CORPUS, CFG)
print("units:", len(CORPUS), "gold:", len(SPLITS["gold"]), "dev:", len(SPLITS["dev"]), "unlabeled:", len(SPLITS["unlabeled"]))
GOLD_TEMPLATE = make_gold_template(ROOT, CFG, CORPUS, SPLITS)
print("gold template:", GOLD_TEMPLATE)
INDEX = build_index(ROOT, CFG, CORPUS)
print("index:", INDEX)
try:
    from google.colab import files
    files.download(str(GOLD_TEMPLATE))
except Exception as exc:
    print("Download skipped:", exc)

## Stop here for human labeling

Fill all `C01_label` ... `C12_label` columns with 0/1 and the matching polarity columns with -1/0/1. Save the file as `results/qual/gold_labels.csv`, upload it into the package, then run the freeze step below. Do not use LLM output to fill the gold set.

**Q2 must run in a fresh runtime** because vLLM pins its own torch stack.

## Q2. Fresh runtime: install vLLM and freeze the human gold set

In [ ]:
!pip -q install vllm

In [ ]:
from pathlib import Path
import sys, json
from google.colab import files
ROOT = Path("/content/loklok_v8")
if not ROOT.exists():
    ROOT = Path("/content/loklok-thesis-final-upgrade")
assert ROOT.exists()
sys.path.insert(0, str(ROOT / "02_code/qual"))
from qual_pipeline import load_config, make_gold_freeze, check_gold, load_generator, code_units, verify_coding
CFG = load_config(ROOT / "02_code/qual/qual_config.yaml"); CFG["_root"] = str(ROOT)
GOLD_PATH = ROOT / CFG["gold_labels"]
if not GOLD_PATH.exists():
    print("Upload gold_labels.csv")
    up = files.upload()
    GOLD_PATH.parent.mkdir(parents=True, exist_ok=True)
    for name, payload in up.items():
        (GOLD_PATH.parent / Path(name).name).write_bytes(payload)
assert GOLD_PATH.exists()
FREEZE = make_gold_freeze(ROOT, CFG)
print("frozen gold:", FREEZE)
GEN = load_generator(CFG)
print("backend:", GEN["backend"])


## Q2. Code with a local model, verify quotes, and export

In [ ]:
import yaml
from qual_pipeline import code_units, verify_coding
CODEBOOK = yaml.safe_load((ROOT / CFG["codebook"]).read_text(encoding="utf-8"))
CODING = code_units(ROOT, CFG, CODEBOOK, GEN, variant="V1")
print("coded units:", len(CODING))
VERIFY = verify_coding(ROOT, CFG, variant="V1")
print("assignments:", len(VERIFY), "exact:", float(VERIFY.exact.mean()), "fuzzy:", float(VERIFY.fuzzy.mean()))
VERIFY.to_csv(ROOT / CFG["output_dir"] / "qual_quote_verification.csv", index=False)

## Q3. Validate against human gold, synthesize, and test prior sensitivity

In [ ]:
import json
from qual_pipeline import validate, synthesize, weight_prior_sensitivity, export_macros, retrieval_audit
VALID = validate(ROOT, CFG, variant="V1")
print(json.dumps(VALID, indent=2))
PREV = synthesize(ROOT, CFG, variant="V1")
print(PREV.head(20) if len(PREV) else "no participant prevalence rows")
RETR = retrieval_audit(ROOT, CFG, variant="V1")
print("retrieval audit:", len(RETR))
PRIOR = weight_prior_sensitivity(ROOT, CFG, VALID)
print(json.dumps(PRIOR, indent=2))
MACROS = export_macros(ROOT, CFG, VALID, PRIOR)
print("macros:", MACROS)
try:
    from google.colab import files
    for p in [ROOT / CFG["output_dir"] / "qual_validation.json", ROOT / CFG["output_dir"] / "qual_prevalence.csv", ROOT / CFG["output_dir"] / "qual_weight_prior.json", MACROS]:
        if p.exists(): files.download(str(p))
except Exception as exc:
    print("Download skipped:", exc)

## Not implemented in this script (requires your decision)

- **E2 persona-trait shifts**: deferred; the trait mapping is not identified by the current 52-person wave.
- **E3 respondent-level theme features**: deferred until the PO provenance gate passes and 52 linked PO entries are coded.
- **E4 local RAG persona agents**: deferred; it is an exploratory extension, not a primary test.
- **Second human coder**: add a 30-unit overlap and report the human?human ceiling; the current script has one gold coder.

The only integration run here is **E1-prior**: it reweights the existing 256-point sweep with validated code evidence. It is labelled as a prior-reweighted sensitivity, not a new ABM run. Do not describe it as an estimated causal weight.